<a href="https://colab.research.google.com/github/awinarko-hue/Latihan/blob/main/IT_Platform/PySpark_EDA_Web_Logs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark EDA untuk Dataset Web Server Log (`logfiles.csv`)

SparkSession, loading data, schema, missing values, DataFrame operations, SQL, agregasi, preprocessing, dan analisis lanjutan—untuk data log HTTP.



## 1. Instalasi dan import

In [1]:
# Jalankan jika PySpark belum terpasang di environment:
# %pip install pyspark

import os
import re
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Membuat SparkSession

In [2]:
spark = (SparkSession.builder
         .appName("PySpark EDA - Web Server Logs")
         .getOrCreate())

print("Spark version:", spark.version)
spark.sparkContext.setLogLevel("WARN")

Spark version: 4.0.4


## 3. Memuat log mentah

Gunakan `spark.read.text`, bukan `read.csv`, karena file log memiliki tanda kutip dan koma di dalam User-Agent. Cara ini mencegah satu baris log terpecah menjadi beberapa kolom.

In [ ]:
# Sesuaikan path ke file log Anda yang sebenarnya.
# Opsi 1: Jika Anda mengunggahnya langsung ke direktori sementara Colab:
# file_path = "/content/logfiles.csv"

# Opsi 2: Jika file berada di Google My Drive Anda (dalam folder utama My Drive):
file_path = "/content/drive/MyDrive/SMT_II/IT Plafform/Tugas2"

raw = spark.read.text(file_path).withColumnRenamed("value", "raw_line")
print("Jumlah baris mentah:", raw.count())
raw.show(5, truncate=False)

Jumlah baris mentah: 2000064
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_line                                                                                                                                                                                                                                                                                   |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|233.223.117.90 - - [27/Dec/2037:12:00:00 +0530] "DELETE /usr/admin HTTP/1.0" 502 4963 "-" "Mozilla/5.0 (Windows 

## 4. Parsing log dan membangun schema

In [ ]:
# Format sumber menyerupai access log Apache, tetapi kutipannya di-escape/digandakan.
# Normalisasi kutipan dilakukan sebelum ekstraksi regex.
normalized = raw.withColumn(
    "line",
    F.regexp_replace(
        F.regexp_replace(
        F.regexp_replace(F.col("raw_line"), r',$', ""),
        r'^"|"$', ""
    ),
        '""', '"'
    )
)

# Grup regex:
# 1 IP, 2 timestamp, 3 method, 4 path, 5 protocol,
# 6 status, 7 bytes, 8 referrer, 9 user-agent, 10 duration
log_pattern = (
    r'^(\S+)\s+\S+\s+\S+\s+\[([^\]]+)\]\s+'
    r'"(\S+)\s+(\S+)\s+(HTTP/[^"]+)"\s+'
    r'(\d{3})\s+(\d+)\s+'
    r'"([^"]*)"\s+"(.*?)"\s+(\d+)\s*$'
)

parsed = normalized
fields = [
    ("ip", 1), ("timestamp_text", 2), ("method", 3), ("path", 4),
    ("protocol", 5), ("status_text", 6), ("bytes_text", 7),
    ("referrer", 8), ("user_agent", 9), ("duration_text", 10)
]
for name, group_num in fields:
    parsed = parsed.withColumn(name, F.regexp_extract("line", log_pattern, group_num))

logs = (parsed
    .withColumn("status", F.col("status_text").cast(IntegerType()))
    .withColumn("bytes", F.col("bytes_text").cast(LongType()))
    .withColumn("duration", F.col("duration_text").cast(DoubleType()))
    .withColumn("timestamp", F.to_timestamp("timestamp_text", "dd/MMM/yyyy:HH:mm:ss Z"))
    .withColumn("is_synthetic", F.lit(False))
    .drop("line", "status_text", "bytes_text", "duration_text")
)

# Baris gagal parsing akan memiliki IP atau method kosong.
logs = logs.withColumn(
    "parse_ok",
    (F.col("ip") != "") & (F.col("method") != "") &
    F.col("status").isNotNull() & F.col("bytes").isNotNull()
)

logs.select("ip", "timestamp", "method", "path", "protocol",
            "status", "bytes", "referrer", "user_agent",
            "duration", "parse_ok").show(10, truncate=False)
print("Baris berhasil diparsing:", logs.filter("parse_ok").count())
print("Baris gagal diparsing:", logs.filter("NOT parse_ok").count())

+---------------+-------------------+------+--------------------+--------+------+-----+---------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+
|ip             |timestamp          |method|path                |protocol|status|bytes|referrer                                           |user_agent                                                                                                                                         |duration|parse_ok|
+---------------+-------------------+------+--------------------+--------+------+-----+---------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+
|233.223.117.90 |2037-12-27 06:30:00|DELETE|/usr/admin          |HTTP/1.0|502   |4

## 5. Pembersihan data dan pemeriksaan schema

In [ ]:
# Tinjau baris yang gagal diparsing sebelum membuangnya.
logs.filter(~F.col("parse_ok")).select("raw_line").show(truncate=False)

# Analisis berikut menggunakan baris yang berhasil diparsing.
df = logs.filter(F.col("parse_ok")).drop("parse_ok")

# String kosong atau '-' pada referrer diperlakukan sebagai nilai tidak tersedia.
df = df.withColumn(
    "referrer",
    F.when((F.col("referrer") == "") | (F.col("referrer") == "-"), None)
     .otherwise(F.col("referrer"))
)

df.printSchema()
print("Jumlah baris valid:", df.count())
print("Kolom:", df.columns)
df.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 6. Memeriksa missing values

In [ ]:
def check_missing(dataframe):
    expressions = []
    for c, dtype in dataframe.dtypes:
        if dtype in ("double", "float"):
            expressions.append(
                F.sum(F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1).otherwise(0)).alias(c)
            )
        else:
            expressions.append(
                F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            )
    return dataframe.select(expressions)

check_missing(df).show(truncate=False)

+--------+---+--------------+------+----+--------+--------+----------+------+-----+--------+---------+------------+
|raw_line|ip |timestamp_text|method|path|protocol|referrer|user_agent|status|bytes|duration|timestamp|is_synthetic|
+--------+---+--------------+------+----+--------+--------+----------+------+-----+--------+---------+------------+
|0       |0  |0             |0     |0   |0       |999601  |0         |0     |0    |0       |0        |0           |
+--------+---+--------------+------+----+--------+--------+----------+------+-----+--------+---------+------------+



## 7. Statistik deskriptif dan ringkasan numerik

In [ ]:
df.describe(["status", "bytes", "duration"]).show()
df.select(
    F.count("*").alias("records"),
    F.avg("bytes").alias("avg_bytes"),
    F.stddev("bytes").alias("std_bytes"),
    F.min("bytes").alias("min_bytes"),
    F.max("bytes").alias("max_bytes"),
    F.avg("duration").alias("avg_duration"),
    F.expr("percentile_approx(duration, 0.5)").alias("median_duration"),
    F.min("duration").alias("min_duration"),
    F.max("duration").alias("max_duration")
).show(truncate=False)

+-------+------------------+-----------------+------------------+
|summary|            status|            bytes|          duration|
+-------+------------------+-----------------+------------------+
|  count|           2000010|          2000010|           2000010|
|   mean|373.84233478832607|4999.490113549432|2500.3944575277123|
| stddev|102.96737968546722|50.04074542974658| 1442.435938502798|
|    min|               200|             4763|               1.0|
|    max|               502|             5233|            5000.0|
+-------+------------------+-----------------+------------------+

+-------+-----------------+-----------------+---------+---------+------------------+---------------+------------+------------+
|records|avg_bytes        |std_bytes        |min_bytes|max_bytes|avg_duration      |median_duration|min_duration|max_duration|
+-------+-----------------+-----------------+---------+---------+------------------+---------------+------------+------------+
|2000010|4999.4901135494

## 8. DataFrame operations: filter, sort, dan kolom turunan

In [ ]:
# Kolom kategori status HTTP
df = (df
    .withColumn("status_class", (F.floor(F.col("status") / 100) * 100).cast("string"))
    .withColumn("is_server_error", F.col("status").between(500, 599))
    .withColumn("is_client_error", F.col("status").between(400, 499))
    .withColumn("is_admin_path", F.col("path").rlike(r"(?i)/admin"))
    .withColumn("path_group", F.regexp_extract("path", r"^(/[^/]+)", 1))
)

# Contoh filter dan sorting
df.filter(F.col("status") >= 500).orderBy(F.desc("duration")).show(truncate=False)
df.filter(F.col("method").isin("PUT", "DELETE")).select(
    "ip", "method", "path", "status", "timestamp"
).show(truncate=False)
df.orderBy(F.desc("bytes")).select("ip", "path", "status", "bytes").show(10, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+--------------------------+------+--------------------+--------+---------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------+-----+--------+-------------------+------------+------------+---------------+---------------+-------------+----------+
|raw_line                                                                                                                                                                                                                                                                                      |ip            |time

## 9. Filter menggunakan Spark SQL

In [ ]:
df.createOrReplaceTempView("web_logs")

spark.sql("""
    SELECT status, COUNT(*) AS total_requests
    FROM web_logs
    GROUP BY status
    ORDER BY total_requests DESC
""").show()

spark.sql("""
    SELECT method, path, status, duration
    FROM web_logs
    WHERE status >= 500 OR upper(method) IN ('PUT', 'DELETE')
    ORDER BY duration DESC
""").show(truncate=False)

+------+--------------+
|status|total_requests|
+------+--------------+
|   502|        287114|
|   304|        286677|
|   403|        286029|
|   500|        285459|
|   200|        285129|
|   404|        285079|
|   303|        284523|
+------+--------------+

+------+--------------------+------+--------+
|method|path                |status|duration|
+------+--------------------+------+--------+
|GET   |/usr/admin/developer|500   |5000.0  |
|DELETE|/usr/admin          |500   |5000.0  |
|DELETE|/usr/admin          |500   |5000.0  |
|POST  |/usr/login          |500   |5000.0  |
|GET   |/usr/admin/developer|500   |5000.0  |
|POST  |/usr/admin          |502   |5000.0  |
|POST  |/usr/login          |500   |5000.0  |
|PUT   |/usr/admin/developer|502   |5000.0  |
|PUT   |/usr/register       |404   |5000.0  |
|DELETE|/usr/login          |500   |5000.0  |
|POST  |/usr/admin          |502   |5000.0  |
|POST  |/usr/admin          |502   |5000.0  |
|PUT   |/usr/register       |404   |5000.0  |

## 10. GroupBy dan agregasi

In [ ]:
# Distribusi status HTTP
status_summary = (df.groupBy("status")
    .agg(F.count("*").alias("requests"),
         F.avg("duration").alias("avg_duration"),
         F.avg("bytes").alias("avg_bytes"))
    .orderBy("status"))
status_summary.show(truncate=False)

# Metode HTTP
df.groupBy("method").agg(
    F.count("*").alias("requests"),
    F.avg("duration").alias("avg_duration"),
    F.sum(F.when(F.col("status") >= 500, 1).otherwise(0)).alias("server_errors")
).orderBy(F.desc("requests")).show(truncate=False)

# Endpoint
df.groupBy("path").agg(
    F.count("*").alias("requests"),
    F.countDistinct("ip").alias("unique_ips"),
    F.avg("duration").alias("avg_duration"),
    F.sum(F.when(F.col("status") >= 500, 1).otherwise(0)).alias("server_errors")
).orderBy(F.desc("requests")).show(truncate=False)

# IP sumber
df.groupBy("ip").agg(
    F.count("*").alias("request_count"),
    F.countDistinct("path").alias("distinct_paths"),
    F.sum(F.when(F.col("status") >= 400, 1).otherwise(0)).alias("errors")
).orderBy(F.desc("request_count")).show(truncate=False)

+------+--------+------------------+-----------------+
|status|requests|avg_duration      |avg_bytes        |
+------+--------+------------------+-----------------+
|200   |285129  |2501.6449151086   |4999.670012520649|
|303   |284523  |2503.5111291530034|4999.409745433585|
|304   |286677  |2499.691792505154 |4999.717309027233|
|403   |286029  |2499.806603526216 |4999.542249212493|
|404   |285079  |2503.930419287285 |4999.388436187864|
|500   |285459  |2498.4383816940435|4999.30458664817 |
|502   |287114  |2495.785228167209 |4999.397727035254|
+------+--------+------------------+-----------------+

+------+--------+------------------+-------------+
|method|requests|avg_duration      |server_errors|
+------+--------+------------------+-------------+
|PUT   |501574  |2497.9071004477905|143836       |
|GET   |501032  |2497.5902896421785|143742       |
|DELETE|499539  |2503.6337663325585|142580       |
|POST  |497865  |2502.472150080845 |142415       |
+------+--------+------------------+-

## 11. Fitur waktu dan tren

In [ ]:
# Catatan: bila timestamp hanya satu waktu, grafik tren tidak informatif.
df_time = (df
    .withColumn("log_date", F.to_date("timestamp"))
    .withColumn("log_hour", F.hour("timestamp"))
    .withColumn("log_minute", F.minute("timestamp"))
)

df_time.groupBy("log_date", "log_hour", "log_minute").agg(
    F.count("*").alias("requests"),
    F.sum(F.when(F.col("status") >= 500, 1).otherwise(0)).alias("server_errors"),
    F.avg("duration").alias("avg_duration")
).orderBy("log_date", "log_hour", "log_minute").show(100, truncate=False)

+----------+--------+----------+--------+-------------+------------------+
|log_date  |log_hour|log_minute|requests|server_errors|avg_duration      |
+----------+--------+----------+--------+-------------+------------------+
|2037-12-27|6       |30        |2000010 |572573       |2500.3944575277123|
+----------+--------+----------+--------+-------------+------------------+



## 12. Indikator anomali berbasis aturan

In [ ]:
# Ini adalah indikator investigasi, bukan bukti serangan.
# Threshold durasi menggunakan persentil dataset agar dapat menyesuaikan data.
q = df.approxQuantile("duration", [0.25, 0.75, 0.95], 0.01)
if len(q) == 3:
    q1, q3, p95 = q
    iqr = q3 - q1
    duration_threshold = q3 + 1.5 * iqr
else:
    duration_threshold = None

df_flags = (df
    .withColumn("flag_server_error", F.col("status").between(500, 599))
    .withColumn("flag_sensitive_method", F.upper("method").isin("PUT", "DELETE"))
    .withColumn("flag_admin_path", F.col("is_admin_path"))
)

if duration_threshold is not None:
    df_flags = df_flags.withColumn(
        "flag_long_duration", F.col("duration") > F.lit(duration_threshold)
    )
else:
    df_flags = df_flags.withColumn("flag_long_duration", F.lit(False))

df_flags = df_flags.withColumn(
    "investigation_score",
    F.col("flag_server_error").cast("int") +
    F.col("flag_sensitive_method").cast("int") +
    F.col("flag_admin_path").cast("int") +
    F.col("flag_long_duration").cast("int")
)

df_flags.filter(F.col("investigation_score") > 0).select(
    "timestamp", "ip", "method", "path", "status", "duration",
    "flag_server_error", "flag_sensitive_method", "flag_admin_path",
    "flag_long_duration", "investigation_score"
).orderBy(F.desc("investigation_score"), F.desc("duration")).show(truncate=False)

print("Threshold durasi untuk peninjauan:", duration_threshold)

+-------------------+---------------+------+--------------------+------+--------+-----------------+---------------------+---------------+------------------+-------------------+
|timestamp          |ip             |method|path                |status|duration|flag_server_error|flag_sensitive_method|flag_admin_path|flag_long_duration|investigation_score|
+-------------------+---------------+------+--------------------+------+--------+-----------------+---------------------+---------------+------------------+-------------------+
|2037-12-27 06:30:00|88.83.142.124  |DELETE|/usr/admin/developer|500   |5000.0  |true             |true                 |true           |false             |3                  |
|2037-12-27 06:30:00|83.44.80.197   |PUT   |/usr/admin/developer|502   |5000.0  |true             |true                 |true           |false             |3                  |
|2037-12-27 06:30:00|83.44.80.197   |PUT   |/usr/admin/developer|502   |5000.0  |true             |true            

## 13. Perluasan data (opsional, data sintetis)

Untuk menguji skalabilitas pipeline, kita dapat membuat variasi sintetis dari record yang ada. Ini **bukan data observasi nyata**, tidak boleh dipakai untuk menyimpulkan pola serangan aktual, dan tidak menggantikan pengumpulan log tambahan dari server. Aktifkan hanya jika diperlukan untuk uji teknis.

In [ ]:
ENABLE_SYNTHETIC_EXPANSION = True
SYNTHETIC_MULTIPLIER = 100  # jumlah salinan sintetis per record sumber

if ENABLE_SYNTHETIC_EXPANSION:
    # Duplikasi baris untuk uji beban pipeline; kolom is_synthetic menandai asalnya.
    # Untuk simulasi yang lebih realistis, buat skenario eksplisit dan variasikan
    # timestamp/status/path secara terkontrol, bukan menganggap salinan sebagai data nyata.
    synthetic = (df
        .withColumn("is_synthetic", F.lit(True))
        .withColumn("raw_line", F.concat(F.col("raw_line"), F.lit(" [SYNTHETIC TEST COPY]")))
        .withColumn("duration", F.col("duration") * F.lit(1.0))
    )
    for _ in range(SYNTHETIC_MULTIPLIER - 1):
        synthetic = synthetic.unionByName(
            df.withColumn("is_synthetic", F.lit(True))
              .withColumn("raw_line", F.concat(F.col("raw_line"), F.lit(" [SYNTHETIC TEST COPY]")))
              .withColumn("duration", F.col("duration") * F.lit(1.0))
        )
    df_expanded = df.unionByName(synthetic)
    print("Record asli:", df.filter(~F.col("is_synthetic")).count())
    print("Record sintetis:", df_expanded.filter(F.col("is_synthetic")).count())
    print("Total:", df_expanded.count())
else:
    df_expanded = df
    print("Perluasan sintetis dinonaktifkan. Total record:", df_expanded.count())

Perluasan sintetis dinonaktifkan. Total record: 2000010


## 14. Machine learning opsional: clustering untuk eksplorasi

In [ ]:
# KMeans hanya untuk eksplorasi pola, bukan klasifikasi serangan.
# Dengan dataset kecil, hasil clustering tidak dapat dianggap valid secara statistik.
ml_df = (df_expanded
    .filter(F.col("status").isNotNull() & F.col("bytes").isNotNull() & F.col("duration").isNotNull())
    .withColumn("method_upper", F.upper("method"))
)

indexer = StringIndexer(
    inputCol="method_upper", outputCol="method_idx", handleInvalid="keep"
)
assembler = VectorAssembler(
    inputCols=["status", "bytes", "duration", "method_idx"],
    outputCol="features_raw",
    handleInvalid="keep"
)
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features",
    withStd=True, withMean=False
)
kmeans = KMeans(k=2, seed=42, featuresCol="features", predictionCol="cluster")

pipeline = Pipeline(stages=[indexer, assembler, scaler, kmeans])
if ml_df.count() >= 2:
    cluster_model = pipeline.fit(ml_df)
    clustered = cluster_model.transform(ml_df)
    clustered.groupBy("cluster").agg(
        F.count("*").alias("records"),
        F.avg("status").alias("avg_status"),
        F.avg("bytes").alias("avg_bytes"),
        F.avg("duration").alias("avg_duration")
    ).show(truncate=False)
else:
    print("Record valid kurang dari 2; clustering dilewati.")

+-------+-------+------------------+-----------------+------------------+
|cluster|records|avg_status        |avg_bytes        |avg_duration      |
+-------+-------+------------------+-----------------+------------------+
|1      |1002606|373.94828078028655|4999.518510760957|2497.7487806775544|
|0      |997404 |373.7358362308553 |4999.461568231128|2503.053933010094 |
+-------+-------+------------------+-----------------+------------------+



## 15. Ekspor hasil yang sudah diproses

In [3]:
# Spark menulis direktori output berisi beberapa part-file.
# Gunakan path output baru agar tidak menimpa hasil sebelumnya.
output_path = "/content/drive/MyDrive/SMT_II/IT Plafform/Tugas2/processed_web_logs_parquet"

# Aktifkan baris berikut jika ingin menyimpan hasil:
# df.write.mode("overwrite").parquet(output_path)
# print("Data tersimpan di:", output_path)

# Alternatif CSV untuk dataset kecil:
# df.coalesce(1).write.mode("overwrite").option("header", True).csv("/mnt/data/processed_web_logs_csv")

## 16. Kesimpulan dan batasan

- Notebook mengikuti alur PySpark pada notebook referensi, tetapi mengganti atribut otomotif dengan atribut log HTTP.
- File dibaca sebagai teks mentah agar koma di User-Agent tidak merusak parsing.
- Periksa `parse_ok` dan contoh `raw_line` jika pola log sumber berubah.
- Pastikan satuan `duration` melalui dokumentasi sumber; notebook tidak mengasumsikan milidetik atau mikrodetik.
- Validasi timestamp, alamat IP, dan definisi kolom sebelum memakai hasil untuk keputusan operasional.
- Data sintetis hanya untuk pengujian teknis. Untuk deteksi anomali yang layak, kumpulkan lebih banyak log lintas waktu dan pertimbangkan label insiden yang telah diverifikasi.

In [4]:
# Akhiri sesi saat semua analisis selesai.

spark.stop()